<a href="https://colab.research.google.com/github/Yash-bebop/ML-Based-Stock-Price-Prediction/blob/main/stock_prediction_pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stock Price Prediction - Full ML Pipeline v2.0
**Upgraded Features:** Agent 1 (Live Data Scraper) - FinBERT Sentiment - FRED Macro - Groq LLM Analysis - Optuna Tuning - SHAP Explainability - Walk-Forward Validation - Multi-Step Forecasting - Confusion Matrix - P-Values - Fixed Backtest - Risk Management - Gradio Frontend

> **All tools used are 100% free.** No paid APIs required.

In [1]:
!pip install yfinance ta xgboost plotly scikit-learn tensorflow seaborn transformers torch optuna shap lightgbm praw fredapi requests beautifulsoup4 groq gradio scipy -q

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, yfinance as yf, json, os, joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
import plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from scipy import stats
from ta.trend import MACD, EMAIndicator, SMAIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb, lightgbm as lgb
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
print('All libraries loaded successfully.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All libraries loaded successfully.


## Configuration

In [2]:
CONFIG = {
    'ticker'          : 'AAPL',
    'start_date'      : '2018-01-01',
    'end_date'        : datetime.today().strftime('%Y-%m-%d'),
    'test_size'       : 0.2,
    'look_back'       : 60,
    'forecast_days'   : 1,
    'random_state'    : 42,
    'n_folds'         : 5,
    'multistep_days'  : [1, 3, 5, 10],
    'initial_capital' : 10_000.0,
    'transaction_cost': 0.001,
}
np.random.seed(CONFIG['random_state'])
tf.random.set_seed(CONFIG['random_state'])
print(f"Config: {CONFIG['ticker']} | {CONFIG['start_date']} to {CONFIG['end_date']}")

Config: AAPL | 2018-01-01 to 2026-06-18


## API Keys (All Free)
Add keys via the Secrets tab in Colab. Never hardcode keys.

| Key Name | Source |
|---|---|
| GROQ_API_KEY | console.groq.com |
| FRED_API_KEY | fred.stlouisfed.org |
| REDDIT_CLIENT_ID | reddit.com/prefs/apps |
| REDDIT_CLIENT_SECRET | reddit.com/prefs/apps |

In [3]:
try:
    from google.colab import userdata
    KEYS = {
        'groq'         : userdata.get('GROQ_API_KEY'),
        'fred'         : userdata.get('FRED_API_KEY'),
        'reddit_id'    : userdata.get('REDDIT_CLIENT_ID'),
        'reddit_secret': userdata.get('REDDIT_CLIENT_SECRET'),
    }
    print('Keys loaded from Colab Secrets')
except Exception:
    KEYS = {'groq': '', 'fred': '', 'reddit_id': '', 'reddit_secret': ''}
    print('Running without Colab Secrets - add keys manually to KEYS dict')

Running without Colab Secrets - add keys manually to KEYS dict


## Step 1 - Raw Data Download

In [4]:
def download_stock_data(ticker, start, end):
    print(f'Downloading {ticker} from Yahoo Finance...')
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty: raise ValueError(f'No data for {ticker}')
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.columns = [c.strip() for c in df.columns]
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True); df.dropna(how='all', inplace=True)
    print(f"Downloaded {len(df)} days | Close range: {df['Close'].min():.2f} to {df['Close'].max():.2f}")
    return df

df_raw = download_stock_data(CONFIG['ticker'], CONFIG['start_date'], CONFIG['end_date'])
print(df_raw.tail(3))

Downloaded 2126 days | Close range: 33.74 to 315.20
                 Close        High         Low        Open    Volume
Date                                                                
2026-06-15  296.420013  297.779999  291.700012  294.119995  45732600
2026-06-16  299.239990  300.480011  293.970001  295.250000  39874400
2026-06-17  295.950012  302.070007  294.359985  300.850006  42685500


## Step 2 - Multi-Ticker Market Context (SPY / QQQ / XLK / VIX)

In [5]:
def add_market_context(df, tickers=('SPY', 'QQQ', 'XLK', '^VIX')):
    enriched = df.copy()
    for t in tickers:
        try:
            col_name = t.replace('^', '')
            mkt = yf.download(t, start=CONFIG['start_date'], end=CONFIG['end_date'],
                              auto_adjust=True, progress=False)['Close']
            if isinstance(mkt, pd.DataFrame): mkt = mkt.squeeze()
            mkt.index = pd.to_datetime(mkt.index)
            enriched[f'{col_name}_Close']  = mkt.reindex(enriched.index, method='ffill')
            enriched[f'{col_name}_Return'] = enriched[f'{col_name}_Close'].pct_change()
            enriched[f'{col_name}_Lag1']   = enriched[f'{col_name}_Return'].shift(1)
            print(f"  {t}: {enriched[f'{col_name}_Return'].notna().sum()} rows")
        except Exception as e:
            print(f'  {t} failed: {e}')
    return enriched

df_raw_enriched = add_market_context(df_raw)
print(f'Shape: {df_raw.shape} -> {df_raw_enriched.shape}')

  SPY: 2125 rows
  QQQ: 2125 rows
  XLK: 2125 rows
  ^VIX: 2125 rows
Shape: (2126, 5) -> (2126, 17)


## Step 3 - Agent 1: Live Data Scraper
- **Yahoo Finance** news (BeautifulSoup, no key)
- **Reddit** posts from r/stocks, r/wallstreetbets, r/investing (PRAW, free key)
- **FRED** macro: VIX, Fed Funds Rate, Yield Curve, CPI (free key)
- **Finviz** analyst ratings (BeautifulSoup, no key)

All text scored by **FinBERT** (runs locally, zero cost).

In [6]:
from transformers import pipeline as hf_pipeline

print('Loading FinBERT...')
try:
    finbert = hf_pipeline('text-classification', model='ProsusAI/finbert',
                           tokenizer='ProsusAI/finbert', device=-1, truncation=True, max_length=512)
    print('FinBERT ready')
except Exception as e:
    finbert = None; print(f'FinBERT unavailable: {e}')

def scrape_yahoo_news(ticker, max_articles=25):
    try:
        r = requests.get(f'https://finance.yahoo.com/quote/{ticker}/news',
                          headers={'User-Agent':'Mozilla/5.0'}, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        headlines = [{'source':'Yahoo','text':t.get_text(strip=True),'score':1}
                     for t in soup.find_all(['h3','h2'])[:max_articles] if len(t.get_text(strip=True))>20]
        print(f'  Yahoo: {len(headlines)} headlines'); return headlines
    except Exception as e: print(f'  Yahoo error: {e}'); return []

def scrape_reddit_sentiment(ticker, max_posts=30):
    if not KEYS['reddit_id']: print('  Reddit: no keys'); return []
    try:
        import praw
        reddit = praw.Reddit(client_id=KEYS['reddit_id'], client_secret=KEYS['reddit_secret'],
                              user_agent='StockBot/1.0')
        posts = [{'source':f'r/{s}','text':p.title,'score':max(p.score,1)}
                 for s in ['stocks','wallstreetbets','investing']
                 for p in reddit.subreddit(s).search(ticker,limit=10,sort='new',time_filter='week')]
        print(f'  Reddit: {len(posts)} posts'); return posts[:max_posts]
    except Exception as e: print(f'  Reddit error: {e}'); return []

def fetch_macro_features(start_date, end_date):
    if not KEYS['fred']: print('  FRED: no key'); return pd.DataFrame()
    try:
        from fredapi import Fred
        f = Fred(api_key=KEYS['fred'])
        series = {'DFF':'Fed_Funds_Rate','T10Y2Y':'Yield_Curve','VIXCLS':'VIX_FRED','CPIAUCSL':'CPI'}
        data = [f.get_series(k,observation_start=start_date,observation_end=end_date).rename(v)
                for k,v in series.items()]
        macro = pd.concat(data, axis=1)
        macro.index = pd.to_datetime(macro.index)
        return macro.resample('B').last().ffill()
    except Exception as e: print(f'  FRED error: {e}'); return pd.DataFrame()

def scrape_analyst_rating(ticker):
    score_map = {'strong buy':2.0,'buy':1.5,'overweight':1.0,'hold':0.0,'neutral':0.0,
                 'underweight':-1.0,'sell':-1.5,'strong sell':-2.0}
    try:
        r = requests.get(f'https://finviz.com/quote.ashx?t={ticker}',
                          headers={'User-Agent':'Mozilla/5.0'}, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        for td in soup.find_all('td'):
            if 'Recom' in td.get_text():
                val = td.find_next_sibling('td')
                if val:
                    label = val.get_text(strip=True).lower()
                    s = score_map.get(label, 0.0); print(f'  Analyst: {label} -> {s}'); return s
    except Exception as e: print(f'  Finviz error: {e}')
    return 0.0

def score_sentiment(texts):
    if not texts or finbert is None: return 0.0
    label_map = {'positive':1.0,'neutral':0.0,'negative':-1.0}
    scores = []
    for item in texts:
        try:
            r = finbert(item['text'][:512])[0]
            scores.append(label_map.get(r['label'],0.0)*r['score']*(1+np.log1p(item.get('score',1))))
        except: pass
    return float(np.mean(scores)) if scores else 0.0

def run_data_agent(df, ticker):
    print(f'\nAGENT 1: {ticker}\n' + '='*55)
    enriched = df.copy()
    yahoo_news = scrape_yahoo_news(ticker); reddit_posts = scrape_reddit_sentiment(ticker)
    news_score = score_sentiment(yahoo_news); reddit_score = score_sentiment(reddit_posts)
    analyst_score = scrape_analyst_rating(ticker)
    enriched['News_Sentiment'] = 0.0; enriched['Reddit_Sentiment'] = 0.0
    enriched['Analyst_Score'] = analyst_score
    enriched.iloc[-1, enriched.columns.get_loc('News_Sentiment')] = news_score
    enriched.iloc[-1, enriched.columns.get_loc('Reddit_Sentiment')] = reddit_score
    macro_df = fetch_macro_features(CONFIG['start_date'], CONFIG['end_date'])
    if not macro_df.empty:
        enriched = enriched.join(macro_df, how='left')
        for col in macro_df.columns: enriched[col] = enriched[col].ffill().bfill()
    report = {
        'ticker': ticker, 'run_timestamp': datetime.now().isoformat(),
        'news_sentiment': round(news_score,4), 'reddit_sentiment': round(reddit_score,4),
        'analyst_score': round(analyst_score,4), 'headlines_count': len(yahoo_news),
        'reddit_count': len(reddit_posts),
    }
    print(f'Agent 1 done. New cols: {[c for c in enriched.columns if c not in df.columns]}')
    return enriched, report

df_raw_enriched, sentiment_report = run_data_agent(df_raw_enriched, CONFIG['ticker'])
print(f'\nSentiment Report:\n{json.dumps(sentiment_report, indent=2)}')

Loading FinBERT...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT ready

AGENT 1: AAPL
  Yahoo: 0 headlines
  Reddit: no keys
  Analyst: 1.94 -> 0.0
  FRED: no key
Agent 1 done. New cols: ['News_Sentiment', 'Reddit_Sentiment', 'Analyst_Score']

Sentiment Report:
{
  "ticker": "AAPL",
  "run_timestamp": "2026-06-18T18:31:24.794887",
  "news_sentiment": 0.0,
  "reddit_sentiment": 0.0,
  "analyst_score": 0.0,
  "headlines_count": 0,
  "reddit_count": 0
}


## Step 4 - Feature Engineering (25+ Features)

In [7]:
def engineer_features(df):
    print('Engineering features...')
    data = df.copy()
    close = data['Close'].squeeze(); high = data['High'].squeeze()
    low = data['Low'].squeeze(); vol = data['Volume'].squeeze()
    data['SMA_20']  = SMAIndicator(close, 20).sma_indicator()
    data['SMA_50']  = SMAIndicator(close, 50).sma_indicator()
    data['SMA_200'] = SMAIndicator(close, 200).sma_indicator()
    data['EMA_12']  = EMAIndicator(close, 12).ema_indicator()
    data['EMA_26']  = EMAIndicator(close, 26).ema_indicator()
    macd = MACD(close)
    data['MACD'] = macd.macd(); data['MACD_Signal'] = macd.macd_signal(); data['MACD_Hist'] = macd.macd_diff()
    data['RSI_14'] = RSIIndicator(close, 14).rsi()
    stoch = StochasticOscillator(high, low, close)
    data['Stoch_K'] = stoch.stoch(); data['Stoch_D'] = stoch.stoch_signal()
    bb = BollingerBands(close, 20, 2)
    data['BB_Upper'] = bb.bollinger_hband(); data['BB_Middle'] = bb.bollinger_mavg()
    data['BB_Lower'] = bb.bollinger_lband(); data['BB_Width'] = bb.bollinger_wband(); data['BB_Pct'] = bb.bollinger_pband()
    data['ATR_14'] = AverageTrueRange(high, low, close, 14).average_true_range()
    data['Daily_Return'] = close.pct_change(); data['Log_Return'] = np.log(close/close.shift(1))
    data['Rolling_Std_20'] = close.rolling(20).std()
    data['OBV'] = OnBalanceVolumeIndicator(close, vol).on_balance_volume()
    data['Volume_SMA_20'] = vol.rolling(20).mean(); data['Volume_Ratio'] = vol / data['Volume_SMA_20']
    for lag in [1,2,3,5,10]: data[f'Close_Lag_{lag}'] = close.shift(lag)
    data['Rolling_Mean_5']  = close.rolling(5).mean(); data['Rolling_Mean_10'] = close.rolling(10).mean()
    data['Price_vs_SMA20']  = (close - data['SMA_20']) / data['SMA_20']
    data['Price_vs_SMA50']  = (close - data['SMA_50']) / data['SMA_50']
    data['Day_of_Week'] = data.index.dayofweek; data['Month'] = data.index.month; data['Quarter'] = data.index.quarter
    data['Target'] = close.shift(-CONFIG['forecast_days'])
    data.dropna(inplace=True)
    print(f'Done: {data.shape[1]} columns, {len(data)} rows')
    return data

df_features = engineer_features(df_raw_enriched)
EXCLUDE = ['Open','High','Low','Close','Volume','Target','SPY_Close','QQQ_Close','XLK_Close','VIX_Close']
feature_cols = [c for c in df_features.columns if c not in EXCLUDE]
print(f'Features ({len(feature_cols)}): {feature_cols}')

Engineering features...
Done: 56 columns, 1926 rows
Features (46): ['SPY_Return', 'SPY_Lag1', 'QQQ_Return', 'QQQ_Lag1', 'XLK_Return', 'XLK_Lag1', 'VIX_Return', 'VIX_Lag1', 'News_Sentiment', 'Reddit_Sentiment', 'Analyst_Score', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Hist', 'RSI_14', 'Stoch_K', 'Stoch_D', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'BB_Width', 'BB_Pct', 'ATR_14', 'Daily_Return', 'Log_Return', 'Rolling_Std_20', 'OBV', 'Volume_SMA_20', 'Volume_Ratio', 'Close_Lag_1', 'Close_Lag_2', 'Close_Lag_3', 'Close_Lag_5', 'Close_Lag_10', 'Rolling_Mean_5', 'Rolling_Mean_10', 'Price_vs_SMA20', 'Price_vs_SMA50', 'Day_of_Week', 'Month', 'Quarter']


## Step 5 - Time-Aware Train/Test Split

In [8]:
def split_data(df, feature_cols, test_size):
    X = df[feature_cols].values; y = df['Target'].values
    idx = int(len(X)*(1-test_size))
    X_train,X_test = X[:idx],X[idx:]; y_train,y_test = y[:idx],y[idx:]
    train_dates = df.index[:idx]; test_dates = df.index[idx:]
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train); X_test_sc = scaler.transform(X_test)
    print(f'Train: {len(X_train):,} | {train_dates[0].date()} to {train_dates[-1].date()}')
    print(f'Test : {len(X_test):,}  | {test_dates[0].date()} to {test_dates[-1].date()}')
    return X_train_sc, X_test_sc, y_train, y_test, scaler, test_dates

X_train, X_test, y_train, y_test, scaler, test_dates = split_data(df_features, feature_cols, CONFIG['test_size'])

Train: 1,540 | 2018-10-16 to 2024-11-27
Test : 386  | 2024-11-29 to 2026-06-16


## Step 6 - Evaluation Framework

In [19]:
def evaluate_model(y_true, y_pred, model_name):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true-y_pred)/(y_true+1e-8)))*100
    r2   = r2_score(y_true, y_pred)
    dir_acc = np.mean(np.sign(np.diff(y_true))==np.sign(np.diff(y_pred)))*100
    _, p_value = stats.wilcoxon(np.abs(y_true-y_pred), np.abs(np.diff(y_true,prepend=y_true[0])), alternative='less')
    rng = np.random.default_rng(42)
    brmses = [np.sqrt(mean_squared_error(y_true[i:=rng.integers(0,len(y_true),len(y_true))],y_pred[i])) for _ in range(500)]
    ci = np.percentile(brmses,[2.5,97.5])
    return {'Model':model_name,'RMSE':round(rmse,4),'RMSE_CI_95':f'[{ci[0]:.4f},{ci[1]:.4f}]',
            'MAE':round(mae,4),'MAPE (%)':round(mape,4),'R2':round(r2,4),
            'Directional Acc %':round(dir_acc,2),'P-Value':round(p_value,4),
            'Sig (p<0.05)':'Yes' if p_value<0.05 else 'No'}

def plot_confusion_matrix(y_true, y_pred, model_name):
    true_dir = (np.diff(y_true)>0).astype(int); pred_dir = (np.diff(y_pred)>0).astype(int)
    cm = confusion_matrix(true_dir, pred_dir)
    fig,ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm, display_labels=['DOWN','UP'], cmap='Blues').plot(ax=ax, values_format='d')
    ax.set_title(f'{model_name} Confusion Matrix'); plt.tight_layout()
    plt.savefig(f"confusion_{model_name.replace(' ','_')}.png", dpi=120, bbox_inches='tight'); plt.show()
    tn,fp,fn,tp = cm.ravel()
    print(f'  Precision: {tp/(tp+fp+1e-8):.2%} | Recall: {tp/(tp+fn+1e-8):.2%}')

print('Evaluation functions ready')

Evaluation functions ready


## Step 7 - Optuna Hyperparameter Tuning (XGBoost + LightGBM)

In [10]:
tscv = TimeSeriesSplit(n_splits=CONFIG['n_folds'])

def xgb_objective(trial):
    p = {'n_estimators':trial.suggest_int('n_estimators',200,800),
         'max_depth':trial.suggest_int('max_depth',3,9),
         'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
         'subsample':trial.suggest_float('subsample',0.6,1.0),
         'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
         'reg_alpha':trial.suggest_float('reg_alpha',1e-4,5.0,log=True),
         'reg_lambda':trial.suggest_float('reg_lambda',1e-4,5.0,log=True),
         'random_state':42,'n_jobs':-1}
    return np.mean([np.sqrt(mean_squared_error(y_train[te],
        xgb.XGBRegressor(**p,verbosity=0).fit(X_train[tr],y_train[tr]).predict(X_train[te])))
        for tr,te in tscv.split(X_train)])

print('Optuna: XGBoost (40 trials)...')
xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=40, show_progress_bar=True)
best_xgb_params = {**xgb_study.best_params,'random_state':42,'n_jobs':-1}
print(f'Best XGBoost: {best_xgb_params}')

def lgb_objective(trial):
    p = {'n_estimators':trial.suggest_int('n_estimators',200,800),
         'max_depth':trial.suggest_int('max_depth',3,9),
         'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
         'num_leaves':trial.suggest_int('num_leaves',20,100),
         'subsample':trial.suggest_float('subsample',0.6,1.0),
         'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
         'random_state':42,'n_jobs':-1,'verbose':-1}
    return np.mean([np.sqrt(mean_squared_error(y_train[te],
        lgb.LGBMRegressor(**p).fit(X_train[tr],y_train[tr]).predict(X_train[te])))
        for tr,te in tscv.split(X_train)])

print('\nOptuna: LightGBM (40 trials)...')
lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=40, show_progress_bar=True)
best_lgb_params = {**lgb_study.best_params,'random_state':42,'n_jobs':-1,'verbose':-1}
print(f'Best LightGBM: {best_lgb_params}')

Optuna: XGBoost (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

Best XGBoost: {'n_estimators': 700, 'max_depth': 3, 'learning_rate': 0.010271947778769155, 'subsample': 0.7465898526386547, 'colsample_bytree': 0.773104364470842, 'reg_alpha': 0.37895903392793767, 'reg_lambda': 0.0007469361477116782, 'random_state': 42, 'n_jobs': -1}

Optuna: LightGBM (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

Best LightGBM: {'n_estimators': 541, 'max_depth': 3, 'learning_rate': 0.15982077814222884, 'num_leaves': 33, 'subsample': 0.7122343526326318, 'colsample_bytree': 0.6890188734322409, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


## Step 8 - Train All Models

In [11]:
predictions, metrics_list = {}, []
def train_eval(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr); pred = model.predict(Xte)
    predictions[name] = pred
    m = evaluate_model(yte, pred, name); metrics_list.append(m)
    print(f"  {name}: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% p={m['P-Value']}")
    return pred

print('Training models...')
y_pred_lr    = train_eval('Linear Regression', LinearRegression(), X_train, X_test, y_train, y_test)
y_pred_ridge = train_eval('Ridge Regression',  Ridge(alpha=1.0),   X_train, X_test, y_train, y_test)

xgb_model = xgb.XGBRegressor(**best_xgb_params, verbosity=0)
xgb_model.fit(X_train, y_train, eval_set=[(X_test,y_test)], verbose=False)
y_pred_xgb = xgb_model.predict(X_test); predictions['XGBoost'] = y_pred_xgb
m = evaluate_model(y_test, y_pred_xgb, 'XGBoost'); metrics_list.append(m)
print(f"  XGBoost: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}%")
xgb_model.save_model('xgboost_stock_model.json')

lgb_model = lgb.LGBMRegressor(**best_lgb_params)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test); predictions['LightGBM'] = y_pred_lgb
m = evaluate_model(y_test, y_pred_lgb, 'LightGBM'); metrics_list.append(m)
print(f"  LightGBM: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}%")
joblib.dump(lgb_model, 'lightgbm_stock_model.pkl')

rf_model = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=5,
    min_samples_leaf=2, max_features='sqrt', random_state=CONFIG['random_state'], n_jobs=-1)
y_pred_rf = train_eval('Random Forest', rf_model, X_train, X_test, y_train, y_test)
joblib.dump(rf_model, 'random_forest_stock_model.pkl')

y_pred_ensemble = y_pred_xgb*0.4 + y_pred_lgb*0.4 + y_pred_rf*0.2
predictions['Ensemble'] = y_pred_ensemble
m = evaluate_model(y_test, y_pred_ensemble, 'Ensemble'); metrics_list.append(m)
print(f"  Ensemble: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}%")
print('\nAll tree models trained.')

Training models...
  Linear Regression: RMSE=4.2856 DirAcc=52.21% p=0.9786
  Ridge Regression: RMSE=4.5933 DirAcc=51.69% p=0.9999
  XGBoost: RMSE=31.6485 DirAcc=52.47%
  LightGBM: RMSE=31.4443 DirAcc=51.43%
  Random Forest: RMSE=33.6521 DirAcc=54.81% p=1.0
  Ensemble: RMSE=31.9372 DirAcc=54.29%

All tree models trained.


## Step 9 - LSTM Deep Learning

In [12]:
def create_sequences(X, y, lb):
    return (np.array([X[i-lb:i] for i in range(lb,len(X))]),
            np.array([y[i] for i in range(lb,len(X))]))

lb = CONFIG['look_back']
X_all_sc = scaler.transform(df_features[feature_cols].values)
y_all = df_features['Target'].values
X_seq, y_seq = create_sequences(X_all_sc, y_all, lb)
sp = int(len(X_seq)*(1-CONFIG['test_size']))
X_lt, X_le = X_seq[:sp], X_seq[sp:]
y_lt, y_le = y_seq[:sp], y_seq[sp:]
lstm_test_dates = df_features.index[lb+sp:]

lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(lb, X_lt.shape[2])),
    BatchNormalization(), Dropout(0.3),
    LSTM(64, return_sequences=True), BatchNormalization(), Dropout(0.2),
    LSTM(32), Dropout(0.2), Dense(16, activation='relu'), Dense(1)
])
lstm_model.compile(optimizer=Adam(1e-3), loss='huber', metrics=['mae'])
lstm_model.summary()
history = lstm_model.fit(X_lt, y_lt, validation_split=0.15, epochs=100, batch_size=32, shuffle=False,
    callbacks=[EarlyStopping(patience=15,restore_best_weights=True),
               ReduceLROnPlateau(factor=0.5,patience=7,min_lr=1e-6)], verbose=1)
y_pred_lstm = lstm_model.predict(X_le).flatten()
predictions['LSTM'] = (y_pred_lstm, lstm_test_dates)
m_lstm = evaluate_model(y_le, y_pred_lstm, 'LSTM'); metrics_list.append(m_lstm)
print(f"LSTM RMSE: {m_lstm['RMSE']} | DirAcc: {m_lstm['Directional Acc %']}%")
lstm_model.save('lstm_stock_model.h5')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        89,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 152,737 (596.63 KB)

 Trainable params: 152,353 (595.13 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 155ms/step - loss: 119.9852 - mae: 120.4852 - val_loss: 201.6947 - val_mae: 202.1947 - learning_rate: 0.0010
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 163ms/step - loss: 115.7974 - mae: 116.2974 - val_loss: 195.5853 - val_mae: 196.0853 - learning_rate: 0.0010
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 145ms/step - loss: 108.1303 - mae: 108.6303 - val_loss: 189.9109 - val_mae: 190.4109 - learning_rate: 0.0010
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 166ms/step - loss: 99.2204 - mae: 99.7204 - val_loss: 176.5281 - val_mae: 177.0281 - learning_rate: 0.0010
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 6s 142ms/step - loss: 87.3070 - mae: 87.8070 - val_loss: 163.7831 - val_mae: 164.2831 - learning_rate: 0.0010
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 140ms/step - loss: 75.0285 - mae: 75.5254 - val_loss: 147.9176 - val_mae: 148.4176 - learning_rate: 0.0010
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 143ms/step - loss: 64.5588 - mae: 65.0578 - val_loss: 

LSTM RMSE: 114.0213 | DirAcc: 48.79%


## Step 10 - Walk-Forward Validation

In [13]:
def walk_forward_validation(df, feature_cols, train_months=24, test_months=3):
    df = df.copy(); df['YM'] = df.index.to_period('M'); periods = df['YM'].unique(); results = []
    for i in range(train_months, len(periods)-test_months, test_months):
        tr_p = periods[i-train_months:i]; te_p = periods[i:i+test_months]
        tr = df[df['YM'].isin(tr_p)]; te = df[df['YM'].isin(te_p)]
        if len(te) < 5: continue
        sc = StandardScaler()
        Xtr = sc.fit_transform(tr[feature_cols]); Xte = sc.transform(te[feature_cols])
        m = xgb.XGBRegressor(**best_xgb_params,verbosity=0).fit(Xtr,tr['Target'])
        row = evaluate_model(te['Target'].values, m.predict(Xte), f'W{i}')
        row['test_start'] = str(te_p[0]); results.append(row)
        print(f"  W{i}: {te_p[0]} RMSE={row['RMSE']} DirAcc={row['Directional Acc %']}%")
    wf = pd.DataFrame(results)
    print(f"\nWF: {len(wf)} windows | Mean RMSE={wf['RMSE'].mean():.4f}+-{wf['RMSE'].std():.4f}")
    return wf

wf_results = walk_forward_validation(df_features, feature_cols)
fig = go.Figure()
fig.add_trace(go.Scatter(x=wf_results['test_start'],y=wf_results['RMSE'],
                          mode='lines+markers',name='RMSE',line=dict(color='#FF9800')))
fig.add_hline(y=wf_results['RMSE'].mean(),line_dash='dash',line_color='white',
               annotation_text=f"Mean={wf_results['RMSE'].mean():.4f}")
fig.update_layout(template='plotly_dark',title='Walk-Forward RMSE',height=400)
fig.write_html('walkforward_rmse.html'); fig.show()

  W24: 2020-10 RMSE=7.8369 DirAcc=44.44%
  W27: 2021-01 RMSE=4.6996 DirAcc=45.0%
  W30: 2021-04 RMSE=2.7642 DirAcc=41.94%
  W33: 2021-07 RMSE=13.2925 DirAcc=44.44%
  W36: 2021-10 RMSE=15.1172 DirAcc=49.21%
  W39: 2022-01 RMSE=5.0338 DirAcc=55.74%
  W42: 2022-04 RMSE=8.1864 DirAcc=40.98%
  W45: 2022-07 RMSE=3.5832 DirAcc=55.56%
  W48: 2022-10 RMSE=3.8298 DirAcc=50.0%
  W51: 2023-01 RMSE=2.5627 DirAcc=55.74%
  W54: 2023-04 RMSE=8.232 DirAcc=44.26%
  W57: 2023-07 RMSE=4.3866 DirAcc=56.45%
  W60: 2023-10 RMSE=2.8621 DirAcc=53.23%
  W63: 2024-01 RMSE=3.3506 DirAcc=53.33%
  W66: 2024-04 RMSE=9.8255 DirAcc=56.45%
  W69: 2024-07 RMSE=16.6277 DirAcc=50.79%
  W72: 2024-10 RMSE=14.3079 DirAcc=60.32%
  W75: 2025-01 RMSE=9.8284 DirAcc=47.46%
  W78: 2025-04 RMSE=12.7045 DirAcc=49.18%
  W81: 2025-07 RMSE=5.032 DirAcc=49.21%
  W84: 2025-10 RMSE=23.8744 DirAcc=63.49%
  W87: 2026-01 RMSE=9.664 DirAcc=51.67%

WF: 22 windows | Mean RMSE=8.5274+-5.5802


## Step 11 - SHAP Explainability

In [14]:
explainer = shap.TreeExplainer(xgb_model); shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(10,8))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, plot_type='bar', show=False)
plt.title('SHAP Feature Importance'); plt.tight_layout()
plt.savefig('shap_summary.png',dpi=150,bbox_inches='tight'); plt.show()
shap.waterfall_plot(shap.Explanation(values=shap_values[-1],base_values=explainer.expected_value,
                                      data=X_test[-1],feature_names=feature_cols),show=True)
plt.figure(figsize=(10,8)); shap.summary_plot(shap_values,X_test,feature_names=feature_cols,show=False)
plt.tight_layout(); plt.savefig('shap_beeswarm.png',dpi=150,bbox_inches='tight'); plt.show()
print('SHAP plots saved')

SHAP plots saved


## Step 12 - Directional Confusion Matrices

In [21]:
def plot_confusion_matrix(y_true, y_pred, model_name):
    true_dir = (np.diff(y_true)>0).astype(int); pred_dir = (np.diff(y_pred)>0).astype(int)
    cm = confusion_matrix(true_dir, pred_dir)
    fig,ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm, display_labels=['DOWN','UP']).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{model_name} Confusion Matrix'); plt.tight_layout()
    plt.savefig(f"confusion_{model_name.replace(' ','_')}.png", dpi=120, bbox_inches='tight'); plt.show()
    tn,fp,fn,tp = cm.ravel()
    print(f'  Precision: {tp/(tp+fp+1e-8):.2%} | Recall: {tp/(tp+fn+1e-8):.2%}')

for name, pred in predictions.items():
    if name == 'LSTM': plot_confusion_matrix(y_le, pred[0], 'LSTM')
    else: plot_confusion_matrix(y_test, pred, name)

  Precision: 54.88% | Recall: 57.56%
  Precision: 54.30% | Recall: 58.54%
  Precision: 55.67% | Recall: 52.68%
  Precision: 54.89% | Recall: 49.27%
  Precision: 57.95% | Recall: 55.12%
  Precision: 57.75% | Recall: 52.68%
  Precision: 51.32% | Recall: 49.49%


## Step 13 - Multi-Step Forecasting (1/3/5/10 days)

In [22]:
def multistep_forecast(df, feature_cols, scaler, horizons=(1,3,5,10)):
    close = df['Close'].squeeze(); X_sc = scaler.transform(df[feature_cols].values)
    sp = int(len(X_sc)*(1-CONFIG['test_size'])); Xtr,Xte = X_sc[:sp],X_sc[sp:]
    res = {'Date':df.index[sp:],'Actual_T1':close.values[sp:]}
    for h in horizons:
        t = close.shift(-h).values; ytr,yte = t[:sp],t[sp:]
        v = ~np.isnan(ytr)
        m = xgb.XGBRegressor(**best_xgb_params,verbosity=0).fit(Xtr[v],ytr[v])
        vt = ~np.isnan(yte); ph = np.full(len(Xte),np.nan); ph[vt] = m.predict(Xte[vt])
        res[f'Pred_T{h}'] = ph
        print(f'  T+{h}: RMSE={np.sqrt(mean_squared_error(yte[vt],ph[vt])):.4f}')
    df_r = pd.DataFrame(res).set_index('Date')
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_r.index,y=df_r['Actual_T1'],name='Actual',line=dict(color='white',width=2)))
    for col,color in zip([f'Pred_T{h}' for h in horizons],['#2196F3','#4CAF50','#FF9800','#F44336']):
        fig.add_trace(go.Scatter(x=df_r.index,y=df_r[col],name=col,line=dict(color=color,dash='dash')))
    fig.update_layout(template='plotly_dark',title='Multi-Step Forecast: 1/3/5/10 Days',height=450)
    fig.write_html('multistep_forecast.html'); fig.show(); return df_r

multistep_df = multistep_forecast(df_features, feature_cols, scaler, horizons=CONFIG['multistep_days'])

  T+1: RMSE=31.5035
  T+3: RMSE=32.3058
  T+5: RMSE=32.5690
  T+10: RMSE=30.2360


## Step 14 - Groq LLM Agent (Free - Llama3)

In [23]:
def run_groq_agent(ticker, metrics_df, sentiment_report, tomorrow_pred, wf_summary):
    s = {'ticker':ticker,'best_model':metrics_df['RMSE'].idxmin(),
         'best_rmse':float(metrics_df['RMSE'].min()),'best_da':float(metrics_df['Directional Acc %'].max()),
         'tm':tomorrow_pred,'sent':sentiment_report,
         'wf_mu':float(wf_summary['RMSE'].mean()),'wf_sd':float(wf_summary['RMSE'].std()),
         'sig':metrics_df[metrics_df['Sig (p<0.05)']=='Yes'].index.tolist()}
    prompt = (f"Senior quant analyst reviewing ML stock prediction.\n\n"
              f"Stock:{s['ticker']} Best:{s['best_model']}(RMSE:{s['best_rmse']:.4f}) "
              f"DirAcc:{s['best_da']:.1f}%\nSig models:{s['sig']}\n"
              f"WF RMSE:{s['wf_mu']:.4f}+-{s['wf_sd']:.4f}\n"
              f"Sentiment: news={s['sent']['news_sentiment']} reddit={s['sent']['reddit_sentiment']} "
              f"analyst={s['sent']['analyst_score']}\n"
              f"Tomorrow: current= xgb= "
              f"ensemble= signal={s['tm']['signal']}\n\n"
              f"Write 5-point analysis: performance, sentiment, confidence, risks, recommendation. "
              f"<300 words, bullets.")
    if not KEYS.get('groq'):
        return (f"LOCAL REPORT: {ticker}\nBest: {s['best_model']} RMSE={s['best_rmse']:.4f}\n"
                f"Signal: {s['tm']['signal']} Ensemble=")
    try:
        from groq import Groq
        resp = Groq(api_key=KEYS['groq']).chat.completions.create(
            model='llama3-8b-8192',messages=[{'role':'user','content':prompt}],
            max_tokens=500,temperature=0.3)
        return resp.choices[0].message.content
    except Exception as e: return f'Groq error: {e}'

print('Groq agent ready')

Groq agent ready


## Step 15 - Model Performance Summary

In [24]:
metrics_df = pd.DataFrame(metrics_list).set_index('Model')
print('\nMODEL PERFORMANCE SUMMARY\n'+'='*80)
print(metrics_df[['RMSE','RMSE_CI_95','MAE','MAPE (%)','R2','Directional Acc %','P-Value','Sig (p<0.05)']].to_string())
print('='*80)
print(f"Best RMSE: {metrics_df['RMSE'].idxmin()} | Best DirAcc: {metrics_df['Directional Acc %'].idxmax()}")

fig,axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle(f"{CONFIG['ticker']} Model Comparison",fontsize=14,fontweight='bold')
palette = ['#2196F3','#9C27B0','#FF9800','#4CAF50','#F44336','#00BCD4','#FF5722']
for ax,metric in zip(axes,['RMSE','MAPE (%)','Directional Acc %']):
    vals = metrics_df[metric].sort_values(ascending=(metric!='Directional Acc %'))
    bars = ax.bar(vals.index,vals.values,color=palette[:len(vals)])
    ax.set_title(metric,fontsize=11); ax.set_xticklabels(vals.index,rotation=20,ha='right',fontsize=8)
    for bar,val in zip(bars,vals.values):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005*max(vals),
                f'{val:.2f}',ha='center',va='bottom',fontsize=8)
    ax.grid(axis='y',alpha=0.3); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.savefig('model_comparison.png',dpi=150,bbox_inches='tight'); plt.show()


MODEL PERFORMANCE SUMMARY
                       RMSE           RMSE_CI_95       MAE  MAPE (%)       R2  Directional Acc %  P-Value Sig (p<0.05)
Model                                                                                                                 
Linear Regression    4.2856      [3.7524,4.7735]    2.9907    1.2655   0.9797              52.21   0.9786           No
Ridge Regression     4.5933      [4.0006,5.1624]    3.2218    1.3670   0.9766              51.69   0.9999           No
XGBoost             31.6485    [29.7749,34.2555]   24.5111    9.3446  -0.1095              52.47   1.0000           No
LightGBM            31.4443    [29.5506,34.0913]   24.1737    9.2058  -0.0952              51.43   1.0000           No
Random Forest       33.6521    [31.7267,36.3250]   26.3156   10.0310  -0.2544              54.81   1.0000           No
Ensemble            31.9372    [30.0476,34.5781]   24.6835    9.4008  -0.1298              54.29   1.0000           No
LSTM               11

## Step 16 - Predict Tomorrow (Live Inference)

In [25]:
def predict_tomorrow(ticker):
    end = datetime.today().strftime('%Y-%m-%d')
    start = (datetime.today()-timedelta(days=400)).strftime('%Y-%m-%d')
    df_l = engineer_features(add_market_context(download_stock_data(ticker, start, end)))
    cc = [c for c in feature_cols if c in df_l.columns]
    lat = df_l[cc].iloc[-1:].values
    if len(cc)<len(feature_cols): lat = np.hstack([lat,np.zeros((1,len(feature_cols)-len(cc)))])
    ls = scaler.transform(lat); cp = float(df_l['Close'].iloc[-1])
    px_xgb = float(xgb_model.predict(ls)[0]); px_lgb = float(lgb_model.predict(ls)[0])
    px_rf  = float(rf_model.predict(ls)[0]); px_ens = px_xgb*0.4+px_lgb*0.4+px_rf*0.2
    pct = (px_ens-cp)/cp*100; direction = 'BULLISH' if px_ens>cp else 'BEARISH'
    result = {'ticker':ticker,'current_price':cp,'xgboost':round(px_xgb,2),
              'lightgbm':round(px_lgb,2),'random_forest':round(px_rf,2),
              'ensemble':round(px_ens,2),'pct_change':round(pct,3),'signal':direction,
              'prediction_date':(datetime.today()+timedelta(days=1)).strftime('%Y-%m-%d')}
    print(f"\n{'='*50}\n{ticker} Tomorrow:  -> Ensemble  ({pct:+.2f}%) [{direction}]\n{'='*50}")
    return result

tomorrow = predict_tomorrow(CONFIG['ticker'])

Downloaded 275 days | Close range: 194.50 to 315.20
  SPY: 274 rows
  QQQ: 274 rows
  XLK: 274 rows
  ^VIX: 274 rows
Engineering features...
Done: 53 columns, 75 rows

AAPL Tomorrow:  -> Ensemble  (-48.34%) [BEARISH]


## Step 17 - Backtest (Stop-Loss, Take-Profit, Transaction Costs)

In [26]:
def backtest_strategy(y_true, y_pred, dates, model_name,
                       cap=10000.0, tc=0.001, sl=0.05, tp=0.10):
    capital,pos,entry = cap,0.0,0.0; eq,bh,trades = [],[],[]
    prices,preds = np.array(y_true),np.array(y_pred); bhs = cap/prices[0]
    for i in range(len(prices)-1):
        cp = prices[i]; sig = 1 if preds[i]>cp else -1
        if pos>0 and entry>0:
            r = (cp-entry)/entry
            if r<=-sl or r>=tp:
                capital=pos*cp*(1-tc); trades.append({'type':'STP','ret':r}); pos,entry=0.0,0.0
        if sig==1 and pos==0:   pos=(capital-capital*tc)/cp; entry=cp; capital=0.0; trades.append({'type':'BUY','ret':None})
        elif sig==-1 and pos>0: r=(cp-entry)/entry; capital=pos*cp*(1-tc); trades.append({'type':'SELL','ret':r}); pos,entry=0.0,0.0
        eq.append(capital+(pos*cp if pos>0 else 0)); bh.append(bhs*cp)
    if pos>0: eq[-1]=pos*prices[-1]*(1-tc)
    equity,bha = np.array(eq),np.array(bh)
    tr=(equity[-1]-cap)/cap*100; br=(bha[-1]-cap)/cap*100
    dr=np.diff(equity)/equity[:-1]; sharpe=(dr.mean()/(dr.std()+1e-10))*np.sqrt(252)
    mdd=((equity-np.maximum.accumulate(equity))/(np.maximum.accumulate(equity)+1e-10)).min()*100
    trets=[t['ret'] for t in trades if t['ret'] is not None]
    wr=sum(r>0 for r in trets)/(len(trets)+1e-10)*100
    print(f'\nBacktest {model_name}: Strategy={tr:+.1f}% BH={br:+.1f}% Sharpe={sharpe:.2f} MDD={mdd:.1f}% Win={wr:.1f}%')
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=list(dates[:len(equity)]),y=equity,name='Strategy',line=dict(color='#4CAF50',width=2)))
    fig.add_trace(go.Scatter(x=list(dates[:len(bha)]),y=bha,name='Buy&Hold',line=dict(color='#2196F3',width=2,dash='dash')))
    fig.add_hline(y=cap,line_dash='dot',line_color='white',opacity=0.5)
    fig.update_layout(template='plotly_dark',title=f'{model_name} vs Buy&Hold Sharpe={sharpe:.2f}',height=420)
    fig.write_html(f"backtest_{model_name.replace(' ','_')}.html"); fig.show()
    return pd.DataFrame({'Date':dates[:len(equity)],'Strategy':equity,'BuyHold':bha})

backtest_df = backtest_strategy(y_test,predictions['Ensemble'],test_dates,'Ensemble',
                                 CONFIG['initial_capital'],CONFIG['transaction_cost'])


Backtest Ensemble: Strategy=+23.5% BH=+25.7% Sharpe=0.78 MDD=-17.3% Win=81.2%


## Step 18 - Full Interactive Dashboard

In [27]:
fig = make_subplots(rows=3,cols=1,row_heights=[0.55,0.25,0.20],shared_xaxes=True,
    subplot_titles=(f"{CONFIG['ticker']} Actual vs Predicted",'Residuals','RSI(14)'),vertical_spacing=0.07)
fig.add_trace(go.Scatter(x=test_dates,y=y_test,name='Actual',line=dict(color='white',width=2.5)),row=1,col=1)
mc={'Linear Regression':'#9C27B0','Ridge Regression':'#FF5722','XGBoost':'#FF9800',
    'LightGBM':'#FFEB3B','Random Forest':'#4CAF50','Ensemble':'#00BCD4'}
for name,pred in predictions.items():
    if name=='LSTM':
        fig.add_trace(go.Scatter(x=lstm_test_dates,y=pred[0],name='LSTM',
                                  line=dict(color='#F44336',dash='dot',width=1.5)),row=1,col=1)
    else:
        fig.add_trace(go.Scatter(x=test_dates,y=pred,name=name,
                                  line=dict(color=mc.get(name,'gray'),dash='dash',width=1.5)),row=1,col=1)
res = y_test-predictions['Ensemble']
fig.add_trace(go.Bar(x=test_dates,y=res,name='Residuals',
                      marker_color=np.where(res>=0,'#4CAF50','#F44336')),row=2,col=1)
rsi = df_features['RSI_14'].loc[test_dates]
fig.add_trace(go.Scatter(x=test_dates,y=rsi,name='RSI14',line=dict(color='#FF9800',width=1.5)),row=3,col=1)
fig.add_hline(y=70,line_dash='dash',line_color='red',row=3,col=1,opacity=0.6)
fig.add_hline(y=30,line_dash='dash',line_color='green',row=3,col=1,opacity=0.6)
fig.update_layout(template='plotly_dark',height=850,
    title=dict(text=f"{CONFIG['ticker']} Full ML Dashboard",font=dict(size=18)),
    hovermode='x unified',legend=dict(orientation='h',y=1.02))
fig.write_html('full_prediction_dashboard.html'); fig.show()

## Step 19 - Groq LLM Report

In [28]:
llm_report = run_groq_agent(CONFIG['ticker'], metrics_df, sentiment_report, tomorrow, wf_results)
print('\n'+'='*60+'\nLLM ANALYSIS REPORT\n'+'='*60+'\n'+llm_report)


LLM ANALYSIS REPORT
LOCAL REPORT: AAPL
Best: Linear Regression RMSE=4.2856
Signal: BEARISH Ensemble=


## Step 20 - Gradio Interactive UI

In [29]:
import gradio as gr

def gradio_predict(ticker_input):
    try:
        r = predict_tomorrow(ticker_input.strip().upper())
        return (f"{r['ticker']} - {r['signal']}\n{'='*40}\n"
                f"Current: \n"
                f"XGBoost:  | LightGBM: \n"
                f"Random Forest: \n"
                f"Ensemble:  ({r['pct_change']:+.2f}%)\n"
                f"For: {r['prediction_date']}")
    except Exception as e: return f'Error: {e}'

with gr.Blocks(theme=gr.themes.Soft(), title='Stock Predictor v2') as demo:
    gr.Markdown('# Stock Price Prediction - ML Pipeline v2.0')
    gr.Markdown("Enter a ticker to get tomorrow's prediction from all trained models.")
    with gr.Row():
        tb  = gr.Textbox(label='Ticker', placeholder='AAPL, MSFT, TSLA...', value=CONFIG['ticker'])
        btn = gr.Button('Predict', variant='primary')
    ob = gr.Textbox(label='Report', lines=12, interactive=False)
    btn.click(fn=gradio_predict, inputs=tb, outputs=ob)
    gr.Markdown('> Educational only. Not financial advice.')

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7f911c566537a0517.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 21 - Save All Artifacts

In [30]:
import zipfile
artifacts = ['xgboost_stock_model.json','lightgbm_stock_model.pkl','random_forest_stock_model.pkl',
             'lstm_stock_model.h5','model_comparison.png','shap_summary.png','shap_beeswarm.png',
             'walkforward_rmse.html','multistep_forecast.html','full_prediction_dashboard.html']
zn = f"{CONFIG['ticker']}_ML_outputs.zip"
with zipfile.ZipFile(zn,'w',zipfile.ZIP_DEFLATED) as zf:
    [zf.write(f) or print(f'  Saved: {f}') if os.path.exists(f) else print(f'  Missing: {f}') for f in artifacts]
print(f'\nZipped: {zn}')
try: from google.colab import files; files.download(zn)
except: print(f'Download manually: {zn}')
print(f"\n{'='*60}\nPIPELINE COMPLETE: {CONFIG['ticker']}\n{'='*60}")
print(f"Best RMSE: {metrics_df['RMSE'].idxmin()} = {metrics_df['RMSE'].min():.4f}")
print(f"Best DirAcc: {metrics_df['Directional Acc %'].idxmax()} = {metrics_df['Directional Acc %'].max():.1f}%")
print(f"Signal: {tomorrow['signal']} | Ensemble:  ({tomorrow['pct_change']:+.2f}%)")
print('='*60)

  Saved: xgboost_stock_model.json
  Saved: lightgbm_stock_model.pkl
  Saved: random_forest_stock_model.pkl
  Saved: lstm_stock_model.h5
  Saved: model_comparison.png
  Saved: shap_summary.png
  Saved: shap_beeswarm.png
  Saved: walkforward_rmse.html
  Saved: multistep_forecast.html
  Saved: full_prediction_dashboard.html

Zipped: AAPL_ML_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PIPELINE COMPLETE: AAPL
Best RMSE: Linear Regression = 4.2856
Best DirAcc: Random Forest = 54.8%
Signal: BEARISH | Ensemble:  (-48.34%)
